# Databricks SQL e Delta Lake - Notebook

In [ ]:
SHOW DATABASES
USE database
SHOW TABLES
SELECT current_database();

CREATE DATABASE DELTABASE
SHOW DATABASES
DROP DATABASE DELTABASE

CREATE SCHEMA DELTABASE
CREATE OR REPLACE TABLE DELTABASE.test(x INT);
SELECT * FROM DELTABASE.test;

DROP SCHEMA DELTABASE
DROP SCHEMA DELTABASE CASCADE

USE default

## Creating Empty Table

In [ ]:
DROP TABLE IF EXISTS t77;

CREATE TABLE t77(
  id INT,
  name STRING
);

SELECT * FROM t77;

## Insert Data

In [ ]:
INSERT INTO t77 VALUES
(1, "Ford"),
(2, "Toyota"),
(3, "Honda");

SELECT * FROM t77;

## Insert Overwrite

In [ ]:
INSERT OVERWRITE t77 VALUES (6, "BMW");

SELECT * FROM t77;

INSERT OVERWRITE t77 VALUES
(1, "BMW"),
(2, "BYD"),
(3, "Chev");

SELECT * FROM t77;

## Basic Utilities

In [ ]:
DESC t77;

DESCRIBE t77;

DESCRIBE EXTENDED t77;

DESCRIBE DETAIL t77;

## Observação

Somente para Hive Store.  
O Unity Catalog protege esse filesystem.

In [ ]:
%fs
ls dbfs:/userhive/warehouse/t77

In [ ]:
DESC HISTORY t77;

## CTAS (Create Table As)

In [ ]:
%fs
ls "dbfs:/databricks-datasets/asa/small/" 

In [ ]:
SELECT * 
FROM CSV.`dbfs:/databricks-datasets/asa/small/`;

In [ ]:
CREATE OR REPLACE TABLE csv_table AS
SELECT *
FROM CSV.`dbfs:/databricks-datasets/asa/small/`;

In [ ]:
DESCRIBE EXTENDED csv_table;

## Observações

Aqui é possível observar:

- `Location` da tabela
- `Type` da tabela (`MANAGED` e `EXTERNAL`)

In [ ]:
%fs
ls dbfs:/user/hive/warehouse/csv_table

## Estrutura esperada

/user/hive/warehouse/csv_table/_delta_log/
/user/hive/warehouse/csv_table/parquets

In [ ]:
DROP TABLE csv_table;

In [ ]:
%fs
ls dbfs:/user/hive/warehouse/csv_table

-- ERROR esperado após DROP TABLE

## Alternative - External Table

In [ ]:
CREATE OR REPLACE TABLE flights
LOCATION "dbfs:/FileStore/flights"
AS
SELECT *
FROM CSV.`dbfs:/databricks-datasets/asa/small/`;

In [ ]:
DESCRIBE EXTENDED flights;

In [ ]:
%fs
ls dbfs:/FileStore/flights

In [ ]:
DROP TABLE flights;

In [ ]:
%fs
ls dbfs:/FileStore/flights

## Observação

Nesse caso não ocorre erro após o DROP TABLE, pois o Hive Metastore ou Unity Catalog não gerencia diretamente esse diretório.

## COPY INTO

In [ ]:
%fs
ls "dbfs:/databricks-datasets/asa/airline/" 

In [ ]:
SELECT *
FROM CSV.`dbfs:/databricks-datasets/asa/airline/1987.csv`;

In [ ]:
CREATE OR REPLACE TABLE new_flights (
  YEAR STRING,
  MONTH STRING,
  DAYOFMONTH STRING,
  DAYOFWEEK STRING,
  DEPTIME STRING,
  CRSDEPTIME STRING,
  ARRTime STRING,
  CRSARRTime STRING,
  UniqueCarrier STRING,
  FlightNum STRING,
  TailNum STRING,
  ActualElapsedTime STRING,
  CRSELapsedTime STRING,
  AirTime STRING,
  ArrDelay STRING,
  DepDelay STRING,
  Origin STRING,
  Dest STRING,
  Distance STRING,
  TaxiIN STRING,
  TaxiOut STRING,
  Cancelled STRING,
  CancellationCode STRING,
  Diverted STRING,
  CarrierDelay STRING,
  WeatherDelay STRING,
  NASDelay STRING,
  SecurityDelay STRING,
  LateAircraftDelay STRING
);

In [ ]:
dbutils.fs.rm("/FileStore/csv", True)
dbutils.fs.mkdirs("/FileStore/csv")
dbutils.fs.cp(
    "dbfs:/databricks-datasets/asa/airline/1987.csv",
    "/FileStore/csv"
)

In [ ]:
COPY INTO new_flights
FROM '/FileStore/csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

In [ ]:
SELECT count(*)
FROM new_flights;

## Observação

Esse formato é útil porque reconhece o que já foi ingerido.

Se o comando abaixo for executado novamente:

COPY INTO new_flights
FROM '/FileStore/csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

O resultado será:

`0 lines affected`

In [ ]:
dbutils.fs.cp(
    "dbfs:/databricks-datasets/asa/airline/1988.csv",
    "/FileStore/csv"
)

In [ ]:
COPY INTO new_flights
FROM '/FileStore/csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

## Resultado esperado

Agora novas linhas serão adicionadas, pois um novo arquivo CSV foi encontrado no diretório.